# Projet 3 : Cell-cell interaction networks inference and structural analysis across cancer types.
## Importation de librarie

In [ ]:
# Importation des libraries
import rds2py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pds
#import libraries
import scanpy as sc
import cell2location as c2l
import squidpy as sq
import commot as ct
import seaborn as sns
import os

In [ ]:
import re

def extract_celltype_names(means_col):
    
    # On retire le préfixe
    prefix = "meanscell_abundance_w_sf_means_per_cluster_mu_fg_"
    celltypes = [col.replace(prefix, "") for col in means_col]
    
    return celltypes

def mean_par_typecell(adata, basename = "sein_ht224p1",threshold_ratio= 0.8):
    def assign_celltypes(row):
        m = row.max()
        return row[row >= threshold_ratio * m].index.tolist()
    
    def mean_celltypes(df):
        df =df.explode("cell_type")
        df["cell_type"] = df["cell_type"].astype(str)
        lr_cols = [c for c in df.columns if c != "cell_type"]
        return df,df.groupby("cell_type")[lr_cols].mean()
    #Récupérer les means d'abondance de cellule
    mat = pds.DataFrame(
        adata_st.obsm['means_cell_abundance_w_sf'],
        index=adata_st.obs_names,
    )
    mat.columns = extract_celltype_names(adata.obsm['means_cell_abundance_w_sf'].columns)
    
    #attribution de type de cellule pour chaque spot
    adata_st.obs["cell_type"]=mat.apply(assign_celltypes, axis=1)
    
    sender = pds.DataFrame(adata.obsm[f"commot-{basename}-sum-sender"].copy())
    receiver = pds.DataFrame(adata.obsm[f"commot-{basename}-sum-receiver"].copy())
    
    sender["cell_type"] =adata.obs["cell_type"].copy()
    receiver["cell_type"] = adata.obs["cell_type"].copy()


    sender,sender_mean = mean_celltypes(sender)
    receiver,receiver_mean = mean_celltypes(receiver)
    return sender, receiver, sender_mean, receiver_mean


def save_mean(sender, receiver, sender_mean, receiver_mean, folder_name, base_dir="ligands_receptors"):

    output_dir = os.path.join(base_dir, folder_name)

    # Vérification + création si nécessaire
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Dossier créé : {output_dir}")
    else:
        print(f"Le dossier existe déjà : {output_dir}")

    #chemins des fichiers
    sender_path = os.path.join(output_dir, "sender.csv")
    sender_mean_path = os.path.join(output_dir, "sender_mean.csv")
    receiver_path = os.path.join(output_dir, "receiver.csv")
    receiver_mean_path = os.path.join(output_dir, "receiver_mean.csv")

    # 3. sauvegarde en CSV
    sender.to_csv(sender_path)
    sender_mean.to_csv(sender_mean_path)
    receiver.to_csv(receiver_path)
    receiver_mean.to_csv(receiver_mean_path)

    print(f"Fichiers sauvegardés dans : {output_dir}")


## Importation de données

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")

#lire les données 
pan224_sc = sc.read("data/data_h5ad/HT224P1-S1.h5ad") # mettre le nom du fichier scRNAseq



pan224_st = sc.read("data/data_h5ad/HT224P1-S1Fc2U1Z1Bs1-SeuratObj.h5ad") # mettre le nom du fichier spatial
pan224_st.obsm['spatial'] = pan224_st.obsm["X_spatial"]
pan224_st.layers["counts"] = pan224_st.X.copy()

print(pan224_sc)
print(pan224_st)

In [ ]:
#vérification de données pour HVG
print("before HVG : ", pan224_sc.shape)
sc.pp.highly_variable_genes(
    pan224_sc,
    n_top_genes=2000,
    subset=False,
    flavor='seurat_v3'
)
print("after HVG : ", pan224_sc.shape)

In [ ]:
shared_features = [
    feature for feature in pan224_st.var_names if feature in pan224_sc.var_names
]
adata_sc = pan224_sc[:, shared_features].copy()
adata_st = pan224_st[:, shared_features].copy()

### Pré-processing

## Déconvolution

### Préparation et entraînement de référence

In [ ]:
#configurer et entrainer sur scRNAseq
c2l.models.RegressionModel.setup_anndata(
    adata=adata_sc,
    batch_key="Piece_ID",
    labels_key="cell_type",
)

model = c2l.models.RegressionModel(adata_sc)

model.view_anndata_setup()

In [ ]:
#Entraînement 
model.train(max_epochs=750, batch_size=2500, train_size=1,
            lr=0.01)

In [ ]:
model.plot_history(20)

In [ ]:
model.plot_QC()

In [ ]:
# Exporter signatures
model.export_posterior(
    adata_sc,
    sample_kwargs={"num_samples": 1000, "batch_size": 2500,},
)

inf_aver = adata_sc.varm['means_per_cluster_mu_fg']  # genes x cell_types
inf_aver.to_csv('signatures_HT224.csv')  # réutilisable

### Déconvolution de données spatiales

In [ ]:
#modele spatial (deconvolution)
c2l.models.Cell2location.setup_anndata(adata_st,layer="counts")
mod_sp = c2l.models.Cell2location(adata_st, cell_state_df=inf_aver,
                                  N_cells_per_location=8)  # ~8 cellules/spot

mod_sp.train(max_epochs=800,lr = 0.01) 
mod_sp.plot_history()  # convergence
mod_sp.plot_QC()  # diagonal

In [ ]:
# Exporter abondances (q05 = confiance haute)
adata_st = mod_sp.export_posterior(adata_st, 
                                   sample_kwargs={'num_samples':1000})
adata_st.obs[mod_sp.factor_names_] = adata_st.obsm['q05_cell_abundance_w_sf']

### sauvegarder des modèles

In [ ]:
model.save("models/ht224p1_sc_regression", overwrite=True)
mod_sp.save("models/ht224p1_spatial", overwrite=True)

### Visualisation de déconvolution

In [ ]:
#visualization et validation

sc.pl.spatial(
    adata_st,
    color=[
        'means_per_cluster_mu_fg_B-cells',
       'means_per_cluster_mu_fg_Macrophages', 'means_per_cluster_mu_fg_Mast',
       'means_per_cluster_mu_fg_Plasma', 'means_per_cluster_mu_fg_T-cells',
       'means_per_cluster_mu_fg_Tumor'
    ],
    spot_size=150
)

sq.pl.spatial_scatter(adata_st, shape = None, 
                      color=['means_per_cluster_mu_fg_T-cells',
                             'means_per_cluster_mu_fg_Tumor'],
                      size = 20) 


# Clustering régions tissulaires
sc.pp.neighbors(adata_st, use_rep='q05_cell_abundance_w_sf')
sc.tl.leiden(adata_st)
sc.tl.umap(adata_st)

In [ ]:
#Visualisation de leiden 
sq.pl.spatial_scatter(
    adata_st,
    color="leiden",
    connectivity_key="spatial_connectivities",
    edges_color="black",
    shape=None,
    edges_width=0.1,
    size=1,
)

## Cell-Cell Communication

In [ ]:
#Chargement de base de données des ligand-récepteurs humains
df_cellchat = ct.pp.ligand_receptor_database(species='human',database='CellChat')
print(df_cellchat.shape)

In [ ]:
df_comm = ct.tl.spatial_communication(adata_st,
    database_name='sein_ht224p1',
    df_ligrec=df_cellchat, dis_thr=500 , heteromeric=True)

### Visualisation de Cell-Cell Communication

In [ ]:
#Visualize signaling level 
pts = adata_st.obsm['spatial']
s = adata_st.obsm['commot-sein_ht224p1-sum-sender']['s-CCL14-CCR1']
r = adata_st.obsm['commot-sein_ht224p1-sum-receiver']['r-CCL14-CCR1']
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].scatter(pts[:,0], pts[:,1], c=s, s=5, cmap='Blues')
ax[0].set_title('Sender')
ax[1].scatter(pts[:,0], pts[:,1], c=r, s=5, cmap='Reds')
ax[1].set_title('Receiver')

### Extraction de données ligands-recepteurs 

In [ ]:
sender, receiver, sender_mean, receiver_mean = mean_par_typecell(adata_st)

In [ ]:
save_mean(sender, receiver, sender_mean, receiver_mean,folder_name = "pancreas_ht224p1")